# 0. Find ID

In [ ]:
from vassar_feetech_servo_sdk import ServoController
import time

PORT = "COM5"
SERVO_TYPE = "sts"

MIN_ID = 0
MAX_ID = 255

controller = ServoController(servo_ids=list(range(MIN_ID, MAX_ID + 1)),
                             servo_type=SERVO_TYPE,
                             port=PORT)
controller.connect()

found = []

for sid in range(MIN_ID, MAX_ID + 1):
    try:
        pos = controller.read_position(sid)
        print(f"ID {sid}: position = {pos}")
        found.append(sid)
    except Exception as e:
       
        # print(f"ID {sid}: no response ({e})")
        pass

    time.sleep(0.01)  # небольшая пауза, чтобы не забивать порт

print("Found servos:", found)

# 1. Position control

In [ ]:
from vassar_feetech_servo_sdk import ServoController
import time

# ================= НАСТРОЙКИ =================
PORT = "COM5"
SERVO_TYPE = "sts"      # "sts" или "hls"
SERVO_ID = 2

# Pinkie
# dist 1 home: 140 <
# prox 2 home: 150 <

# Index
# dist 3 home: 210 <
# prox 6 home: 145 <

# Thumb
# dist 5 home: 210 <
# prox 4 home: 168 <
# rot  7 home: 310 <


q_pos = 150.0             # <<< НУЖНЫЙ УГОЛ В ГРАДУСАХ
SPEED = 60              # медленно и безопасно
ACCEL = 30
# ============================================

def deg_to_ticks(angle_deg):
    """0..360° -> 0..4095"""
    angle_deg = max(0.0, min(360.0, angle_deg))
    return int(angle_deg / 360.0 * 4095)

def ticks_to_deg(ticks):
    """0..4095 -> 0..360°"""
    return ticks / 4095.0 * 360.0

# Подключение к сервоприводу
controller = ServoController(
    servo_ids=[SERVO_ID],
    servo_type=SERVO_TYPE,
    port=PORT
)
controller.connect()

try:
    # Перевод желаемого угла в тики
    ticks = deg_to_ticks(q_pos)
    print(f"Command: {q_pos:.1f}° -> {ticks} ticks")

    # Отправляем команду
    res = controller.write_position(
        {SERVO_ID: ticks},
        speed=SPEED,
        acceleration=ACCEL
    )
    print("Write result:", res)

    # Даём мотору доехать
    time.sleep(2.0)

    # Чтение фактической позиции
    current_ticks = controller.read_position(SERVO_ID)
    current_deg = ticks_to_deg(current_ticks)
    print(f"Motor actual position: {current_deg:.1f}° -> {current_ticks} ticks")

finally:
    controller.disconnect()


# 2. Serial control

In [ ]:
#!/usr/bin/env python3
"""
multi_servo_motion_planner_clamp_offsets.py

Версия с устойчивым открытием порта и резервным dry-run режимом.
Исправление: безопасный парсинг аргументов через parse_known_args() — полезно при запуске в Jupyter.
"""

from typing import Dict, Iterable, List, Optional, Tuple
import time
import argparse
import sys

# Попробуем импортировать SDK и pyserial; при отсутствии — будем предлагать dry-run.
try:
    from vassar_feetech_servo_sdk import ServoController  # type: ignore
    SDK_AVAILABLE = True
except Exception:
    ServoController = None  # type: ignore
    SDK_AVAILABLE = False

# pyserial для проверки доступных COM-портов
try:
    import serial  # type: ignore
    from serial.tools import list_ports  # type: ignore
    PYSERIAL_AVAILABLE = True
except Exception:
    serial = None  # type: ignore
    list_ports = None  # type: ignore
    PYSERIAL_AVAILABLE = False

# ----------------- НАСТРОЙКИ (можно переопределить через CLI) -----------------
PORT = "COM5"
SERVO_TYPE = "sts"           # "sts" или "hls"
SERVO_IDS: List[int] = [1, 2, 3, 4, 5, 6, 7]

SPEED = 60                   # скорость движения (SDK-зависимо)
ACCEL = 30                   # ускорение

TOLERANCE_DEG = 2.0          # погрешность (в градусах), считаем "в позиции"
POLL_INTERVAL = 0.25         # интервал опроса позиций, с
TIMEOUT_S = 8.0              # таймаут ожидания достижения позиции, с
# -------------------------------------------------------------------------------

INITIAL_POSITIONS_DEG: Dict[int, float] = {
    1: 140.0,
    2: 150.0,
    3: 210.0,
    4: 168.0,
    5: 210.0,
    6: 145.0,
    7: 310.0,
}

RELATIVE_LIMITS: Dict[int, Tuple[float, float]] = {
    1: (-90.0, 90.0),
    2: (-90.0, 60.0),
    3: (-90.0, 60.0),
    4: (-90.0, 90.0),
    5: (-90.0, 60.0),
    6: (-90.0, 60.0),
    7: (-180.0, 0.0),
}

# ----------------- MOCK: Fake controller для dry-run / отладки -----------------
class FakeServoController:
    """Простейший имитатор контроллера для dry-run: хранит позиции в градусах."""
    def __init__(self, servo_ids: List[int], servo_type: str = "sts", port: Optional[str] = None):
        self.servo_ids = list(servo_ids)
        self.servo_type = servo_type
        self.port = port
        self._positions = {sid: INITIAL_POSITIONS_DEG.get(sid, 0.0) for sid in self.servo_ids}
        self._connected = False

    def connect(self):
        print("[FAKE] Connecting (dry-run). No hardware involved.")
        self._connected = True

    def disconnect(self):
        print("[FAKE] Disconnecting (dry-run).")
        self._connected = False

    def write_position(self, ticks_map: Dict[int, int], speed: int = 0, acceleration: int = 0):
        for sid, ticks in ticks_map.items():
            deg = float(ticks) / 4095.0 * 360.0
            self._positions[sid] = deg
        return {"result": "ok", "written": list(ticks_map.keys())}

    def read_position(self, sid: int) -> Optional[int]:
        if sid not in self._positions:
            return None
        deg = self._positions[sid]
        ticks = int(round(max(0.0, min(360.0, deg)) / 360.0 * 4095))
        return ticks

# ----------------- УТИЛИТЫ ПРЕОБРАЗОВАНИЯ -----------------
def clamp_deg(angle_deg: float) -> float:
    return max(0.0, min(360.0, float(angle_deg)))

def deg_to_ticks(angle_deg: float) -> int:
    a = clamp_deg(angle_deg)
    return int(round(a / 360.0 * 4095))

def ticks_to_deg(ticks: int) -> float:
    return float(ticks) / 4095.0 * 360.0

def build_ticks_map(positions_deg: Dict[int, float]) -> Dict[int, int]:
    return {sid: deg_to_ticks(deg) for sid, deg in positions_deg.items()}

# ----------------- ВЗАИМОДЕЙСТВИЕ С СЕРВО -----------------
def safe_read_position(controller, servo_id: int) -> Optional[float]:
    try:
        ticks = controller.read_position(servo_id)
        if ticks is None:
            return None
        return ticks_to_deg(int(ticks))
    except Exception as e:
        print(f"[WARN] read_position({servo_id}) failed: {e}")
        return None

def read_positions_batch(controller, ids: Iterable[int]) -> Dict[int, Optional[float]]:
    return {sid: safe_read_position(controller, sid) for sid in ids}

def write_positions(controller, positions_deg: Dict[int, float], speed: int, accel: int) -> bool:
    ticks_map = build_ticks_map(positions_deg)
    print("Sending positions (deg -> ticks):")
    for sid in sorted(ticks_map.keys()):
        print(f"  ID {sid:>2}: {positions_deg[sid]:6.1f}° -> {ticks_map[sid]:4d} ticks")

    try:
        res = controller.write_position(ticks_map, speed=speed, acceleration=accel)
        print("write_position result:", res)
        return True
    except Exception as e:
        print("[ERROR] write_position exception:", e)
        return False

def wait_until_reached(controller,
                       target_positions_deg: Dict[int, float],
                       tol_deg: float,
                       timeout_s: float,
                       poll_interval: float) -> bool:
    start = time.time()
    target_ids = sorted(target_positions_deg.keys())
    reached = {sid: False for sid in target_ids}

    while True:
        current = read_positions_batch(controller, target_ids)

        for sid in target_ids:
            cur = current.get(sid)
            if cur is None:
                reached[sid] = False
            else:
                reached[sid] = abs(cur - target_positions_deg[sid]) <= tol_deg

        elapsed = time.time() - start

        status_parts = []
        for sid in target_ids:
            if reached[sid]:
                status_parts.append(f"{sid}:OK")
            else:
                cur = current.get(sid)
                status_parts.append(f"{sid}:{'?' if cur is None else f'{cur:.1f}°'}")
        print(f"[t={elapsed:4.1f}s] " + "  ".join(status_parts))

        if all(reached.values()):
            print("All servos reached target positions.")
            return True

        if elapsed >= timeout_s:
            print(f"[TIMEOUT] Not all servos reached targets within {timeout_s} s.")
            for sid in target_ids:
                print(f"  ID {sid}: target {target_positions_deg[sid]:6.1f}°, current {current.get(sid)}")
            return False

        time.sleep(poll_interval)

# ----------------- MOTION PLANNER -----------------
class MotionPlanner:
    def __init__(self,
                 controller,
                 servo_ids: List[int],
                 initial_positions_deg: Dict[int, float],
                 relative_limits: Dict[int, Tuple[float, float]],
                 speed: int = SPEED,
                 accel: int = ACCEL,
                 tol_deg: float = TOLERANCE_DEG,
                 poll_interval: float = POLL_INTERVAL,
                 timeout_s: float = TIMEOUT_S):
        self.controller = controller
        self.servo_ids = list(servo_ids)
        self.initial_positions_deg = dict(initial_positions_deg)
        self.relative_limits = dict(relative_limits)

        missing_initial = [sid for sid in self.servo_ids if sid not in self.initial_positions_deg]
        if missing_initial:
            raise ValueError(f"Initial positions missing for servo IDs: {missing_initial}")

        missing_limits = [sid for sid in self.servo_ids if sid not in self.relative_limits]
        if missing_limits:
            raise ValueError(f"Relative limits missing for servo IDs: {missing_limits}")

        self.speed = speed
        self.accel = accel
        self.tol_deg = tol_deg
        self.poll_interval = poll_interval
        self.timeout_s = timeout_s

        self._queue: List[Dict[int, float]] = []

    def _clamp_offset_for_id(self, sid: int, offset: float) -> Tuple[float, bool, float, float]:
        min_off, max_off = self.relative_limits[sid]
        if offset < min_off:
            return min_off, True, min_off, max_off
        if offset > max_off:
            return max_off, True, min_off, max_off
        return offset, False, min_off, max_off

    def go_pos(self, *offsets_deg: float) -> None:
        if len(offsets_deg) != len(self.servo_ids):
            raise ValueError(f"go_pos expects {len(self.servo_ids)} values (one per servo). Got {len(offsets_deg)}.")

        offsets_map: Dict[int, float] = {}
        for i, sid in enumerate(self.servo_ids):
            requested = float(offsets_deg[i])
            clamped, was_clamped, min_lim, max_lim = self._clamp_offset_for_id(sid, requested)
            offsets_map[sid] = clamped
            if was_clamped:
                print(f"[CLAMP] Servo ID {sid}: requested {requested}° -> clamped to {clamped}° (limits [{min_lim}, {max_lim}])")

        self._queue.append(offsets_map)
        print(f"[PLANNER] Queued relative (clamped) target: {offsets_map}")

    def clear_queue(self) -> None:
        self._queue.clear()
        print("[PLANNER] Queue cleared.")

    def _relative_to_absolute(self, offsets_map: Dict[int, float]) -> Dict[int, float]:
        abs_map: Dict[int, float] = {}
        for sid, offset in offsets_map.items():
            base = self.initial_positions_deg[sid]
            abs_deg = clamp_deg(base + offset)
            abs_map[sid] = abs_deg
        return abs_map

    def execute_plan(self) -> Tuple[bool, int]:
        total = len(self._queue)
        print(f"[PLANNER] Executing plan with {total} steps...")
        completed = 0

        for i, offsets in enumerate(self._queue, start=1):
            print(f"\n[PLANNER] Step {i}/{total}: relative offsets = {offsets}")
            target_abs = self._relative_to_absolute(offsets)
            print(f"[PLANNER] Step {i}/{total}: absolute targets = {target_abs}")

            ok = write_positions(self.controller, target_abs, speed=self.speed, accel=self.accel)
            if not ok:
                print(f"[PLANNER] write_positions failed on step {i}. Aborting plan.")
                return False, completed

            reached = wait_until_reached(self.controller,
                                         target_positions_deg=target_abs,
                                         tol_deg=self.tol_deg,
                                         timeout_s=self.timeout_s,
                                         poll_interval=self.poll_interval)
            if not reached:
                print(f"[PLANNER] Step {i} timed out or failed. Aborting plan.")
                return False, completed

            completed += 1
            print(f"[PLANNER] Step {i} completed successfully.")

        print(f"[PLANNER] Plan finished. Completed {completed}/{total} steps.")
        self.clear_queue()
        return True, completed

# ----------------- ВСПОМОГАТЕЛЬНЫЕ ФУНКЦИИ ДЛЯ ПОРТОВ -----------------
def list_available_ports() -> List[str]:
    if not PYSERIAL_AVAILABLE:
        return []
    ports = list_ports.comports()
    return [p.device for p in ports]

def select_port(preferred: Optional[str] = None) -> Optional[str]:
    if not PYSERIAL_AVAILABLE:
        return preferred
    available = list_available_ports()
    if not available:
        return None
    if preferred and preferred in available:
        return preferred
    if preferred and preferred not in available:
        print(f"[WARN] Preferred port {preferred} not found among available: {available}. Will try first available port.")
    return available[0]

# ----------------- MAIN -----------------
def main():
    parser = argparse.ArgumentParser(description="Motion planner with clamped relative offsets and robust port handling.")
    parser.add_argument("--port", type=str, default=PORT, help="COM port to use (e.g. COM5)")
    parser.add_argument("--dry-run", action="store_true", help="Run without hardware (use fake controller).")
    parser.add_argument("--list-ports", action="store_true", help="Just list available COM ports and exit.")

    # ВАЖНО: использовать parse_known_args чтобы игнорировать аргументы, добавляемые IPython/Jupyter
    args, unknown = parser.parse_known_args()
    if unknown:
        # Показываем предупреждение, но не падаем — это нормально при запуске в Jupyter.
        print(f"[WARN] Ignoring unknown CLI args: {unknown}")

    if args.list_ports:
        if not PYSERIAL_AVAILABLE:
            print("pyserial not available; cannot list ports.")
            sys.exit(1)
        ports = list_available_ports()
        print("Available COM ports:", ports)
        sys.exit(0)

    chosen_port = args.port
    if not args.dry_run and PYSERIAL_AVAILABLE:
        sel = select_port(chosen_port)
        if sel is None:
            print("[ERROR] No serial ports detected on the system.")
            print(" - Проверьте подключение кабеля, драйвер CH340/FTDI (Windows Device Manager),")
            print(" - Убедитесь, что порт не занят другим приложением.")
            print(" - Можете запустить с --dry-run для отладки без устройства.")
            sys.exit(1)
        if sel != chosen_port:
            print(f"[INFO] Preferred port '{chosen_port}' not available; using '{sel}' instead.")
        chosen_port = sel

    if args.dry_run:
        controller = FakeServoController(servo_ids=SERVO_IDS, servo_type=SERVO_TYPE, port=chosen_port)
    else:
        if not SDK_AVAILABLE:
            print("[ERROR] vassar_feetech_servo_sdk не доступен в текущем окружении.")
            print("Установите пакет или используйте --dry-run для отладки без оборудования.")
            sys.exit(1)
        controller = ServoController(servo_ids=SERVO_IDS, servo_type=SERVO_TYPE, port=chosen_port)

    print("Connecting to", chosen_port, "...")
    try:
        controller.connect()
    except Exception as e:
        print("[ERROR] controller.connect() failed:", repr(e))
        if PYSERIAL_AVAILABLE:
            ports = list_available_ports()
            print("Available COM ports:", ports)
        print("Suggestions:")
        print(" - Проверьте кабель и питание устройства.")
        print(" - Проверьте драйвер CH340/FTDI (Device Manager).")
        print(" - Убедитесь, что порт не занят другим приложением (IDE, терминал, другой процесс).")
        print(" - Если вы на Windows, попробуйте указать другой COM порт или запустить с --dry-run.")
        sys.exit(1)

    try:
        planner = MotionPlanner(controller=controller,
                                servo_ids=SERVO_IDS,
                                initial_positions_deg=INITIAL_POSITIONS_DEG,
                                relative_limits=RELATIVE_LIMITS,
                                speed=SPEED,
                                accel=ACCEL,
                                tol_deg=TOLERANCE_DEG,
                                poll_interval=POLL_INTERVAL,
                                timeout_s=TIMEOUT_S)

        planner.go_pos(0, 0, 0, 0, 0, 0, 0)
        planner.go_pos(-10, -10, -10, 10, -10, -10, -30)
        planner.go_pos(-20, -20, -20, 20, -20, -20, -60)
        planner.go_pos(-30, -30, -30, 30, -30, -30, -90)
        planner.go_pos(-40, -40, -40, 40, -40, -40, -120)
        planner.go_pos(0, 0, 0, 0, 0, 0, 0)
        planner.go_pos(-60, 0, 0, 0, 0, 0, -180)
        planner.go_pos(0, -60, 0, 0, 0, 0, -180)
        planner.go_pos(0, 0, -60, 0, 0, 0, -180)
        planner.go_pos(0, 0, 0, 0, 0, -60, -180)
        planner.go_pos(0, 0, 0, 0, -60, 0, -180)
        planner.go_pos(0, 0, 0, -60, 0, 0, -180)
        planner.go_pos(0, 0, 0, 0, 0, 0, 0)

        all_ok, completed = planner.execute_plan()
        if not all_ok:
            print(f"Plan aborted after {completed} steps.")
        else:
            print("Plan executed successfully.")

        final_positions = read_positions_batch(controller, SERVO_IDS)
        print("\nFinal positions (deg):")
        for sid in sorted(SERVO_IDS):
            print(f"  ID {sid}: {final_positions.get(sid)}")

    finally:
        print("Disconnecting...")
        try:
            controller.disconnect()
        except Exception as e:
            print("Error during disconnect:", e)

if __name__ == "__main__":
    main()


# 3. GUI hand control

In [2]:
#!/usr/bin/env python3
"""
servo_gui_sequence_with_home_fixed_step_only_layouted_timeout_reset.py

Updated GUI with:
 - single-step execution
 - grouped sliders
 - central vertical buttons column
 - right column: sequence Treeview + log
 - STEP_TIMEOUT_S set to 4 seconds
 - on timeout while waiting for a step, the execution pointer (_exec_next_index)
   is reset to 0 so subsequent Execute starts from the first step.
"""

from __future__ import annotations
import time
import threading
from typing import Dict, Iterable, List, Optional, Tuple
import tkinter as tk
from tkinter import ttk

# ---------------- USER CONFIG ----------------
DEFAULT_PORT = "COM5"
SERVO_IDS: List[int] = [1, 2, 3, 4, 5, 6, 7]
# UI slider order requested by user
SERVO_ORDER: List[int] = [1, 2, 3, 6, 5, 4, 7]
INITIAL_POSITIONS_DEG: Dict[int, float] = {
    1: 140.0, 2: 150.0, 3: 210.0, 4: 168.0, 5: 210.0, 6: 145.0, 7: 310.0
}
RELATIVE_LIMITS: Dict[int, Tuple[float, float]] = {
    1: (-90.0, 90.0), 2: (-90.0, 60.0), 3: (-90.0, 60.0),
    4: (-90.0, 90.0), 5: (-90.0, 60.0), 6: (-90.0, 60.0),
    7: (-180.0, 0.0)
}
DEFAULT_SPEED = 60
DEFAULT_ACCEL = 30
GUI_POLL_MS = 600             # ms
SLIDER_DEBOUNCE_MS = 80       # ms
STEP_TOLERANCE_DEG = 2.0      # tolerance to consider reached
STEP_TIMEOUT_S = 4.0          # timeout per step (changed to 4 seconds)
# ---------------------------------------------

# Phalange names mapping (user provided)
PHALANGE_NAMES: Dict[int, str] = {
    1: "Index Distal",
    2: "Index Proximal",
    3: "Pinkie Distal",
    6: "Pinkie Proximal",
    5: "Thumb Distal",
    4: "Thumb Proximal",
    7: "Index Rotation"
}

# Try to import real SDK (optional)
try:
    from vassar_feetech_servo_sdk import ServoController  # type: ignore
    SDK_AVAILABLE = True
except Exception:
    ServoController = None  # type: ignore
    SDK_AVAILABLE = False

# Try pyserial to list ports
try:
    from serial.tools import list_ports  # type: ignore
    PYSERIAL = True
except Exception:
    list_ports = None  # type: ignore
    PYSERIAL = False

# ---------------- Utilities ----------------
def clamp_deg(angle_deg: float) -> float:
    return max(0.0, min(360.0, float(angle_deg)))


def deg_to_ticks(angle_deg: float) -> int:
    a = clamp_deg(angle_deg)
    return int(round(a / 360.0 * 4095))


def ticks_to_deg(ticks: int) -> float:
    return float(ticks) / 4095.0 * 360.0


def build_ticks_map(positions_deg: Dict[int, float]) -> Dict[int, int]:
    return {sid: deg_to_ticks(deg) for sid, deg in positions_deg.items()}


def safe_read_position(controller, servo_id: int) -> Optional[float]:
    try:
        ticks = controller.read_position(servo_id)
        if ticks is None:
            return None
        return ticks_to_deg(int(ticks))
    except Exception:
        try:
            if hasattr(controller, "read_positions_batch"):
                m = controller.read_positions_batch([servo_id])
                v = m.get(servo_id)
                if v is not None:
                    return ticks_to_deg(int(v))
        except Exception:
            pass
    return None


def read_positions_batch(controller, ids: Iterable[int]) -> Dict[int, Optional[float]]:
    return {sid: safe_read_position(controller, sid) for sid in ids}


def list_available_ports() -> List[str]:
    if not PYSERIAL:
        return []
    ports = list_ports.comports()
    return [p.device for p in ports]

# ----------------- wait util (checks stop_event) -----------------
def wait_until_reached(controller, target_positions_deg: Dict[int, float],
                       tol_deg: float, timeout_s: float, poll_interval: float,
                       stop_event: threading.Event) -> bool:
    """Poll positions until all within tol_deg or timeout/stop_event."""
    start = time.time()
    target_ids = sorted(target_positions_deg.keys())
    reached = {sid: False for sid in target_ids}

    while True:
        if stop_event.is_set():
            return False
        current = read_positions_batch(controller, target_ids)
        for sid in target_ids:
            cur = current.get(sid)
            if cur is None:
                reached[sid] = False
            else:
                reached[sid] = abs(cur - target_positions_deg[sid]) <= tol_deg
        if all(reached.values()):
            return True
        if time.time() - start >= timeout_s:
            return False
        time.sleep(poll_interval)

# ---------------- GUI -----------------
class ServoGUI(tk.Tk):
    def __init__(self):
        super().__init__()
        self.title("Servo GUI — grouped sliders + single-step")
        self.geometry("1400x700")
        self.resizable(True, True)

        # controller / connection
        self.controller = None
        self.connected = False

        # UI state
        self.port_var = tk.StringVar(value=DEFAULT_PORT)
        self.type_var = tk.StringVar(value="sts")
        self.speed_var = tk.IntVar(value=DEFAULT_SPEED)
        self.accel_var = tk.IntVar(value=DEFAULT_ACCEL)
        self.auto_poll = tk.BooleanVar(value=True)

        # per-servo vars
        self.offset_vars: Dict[int, tk.DoubleVar] = {}
        self.abs_labels: Dict[int, tk.Label] = {}
        self.cur_labels: Dict[int, tk.Label] = {}
        self.scales: Dict[int, tk.Scale] = {}

        # debounce ids
        self._debounce_after_ids: Dict[int, Optional[str]] = {sid: None for sid in SERVO_IDS}

        # sequence
        self.sequence: List[Dict[int, float]] = []

        # execution
        self._exec_thread: Optional[threading.Thread] = None
        self._exec_stop_event = threading.Event()
        self._exec_lock = threading.Lock()
        self._is_executing = False
        self._exec_next_index: int = 0

        # build UI
        self._build_ui()
        self._refresh_ports()
        self.after(GUI_POLL_MS, self._periodic_poll)

    def _build_ui(self):
        # Top bar (port, type, speed, accel, connect)
        top = ttk.Frame(self)
        top.pack(side=tk.TOP, fill=tk.X, padx=8, pady=6)

        ttk.Label(top, text="Port:").pack(side=tk.LEFT)
        self.port_combo = ttk.Combobox(top, textvariable=self.port_var, width=12)
        self.port_combo.pack(side=tk.LEFT, padx=(4, 8))
        ttk.Button(top, text="List ports", command=self._refresh_ports).pack(side=tk.LEFT, padx=(0, 8))

        ttk.Label(top, text="Type:").pack(side=tk.LEFT)
        ttk.OptionMenu(top, self.type_var, self.type_var.get(), "sts", "hls").pack(side=tk.LEFT, padx=(0, 12))

        ttk.Label(top, text="Speed:").pack(side=tk.LEFT)
        ttk.Entry(top, textvariable=self.speed_var, width=5).pack(side=tk.LEFT, padx=(0, 8))

        ttk.Label(top, text="Accel:").pack(side=tk.LEFT)
        ttk.Entry(top, textvariable=self.accel_var, width=5).pack(side=tk.LEFT, padx=(0, 12))

        ttk.Button(top, text="Connect (Real)", command=self._connect_real).pack(side=tk.LEFT, padx=(6, 6))
        ttk.Button(top, text="Disconnect", command=self._disconnect).pack(side=tk.LEFT, padx=(0, 12))

        self.status_label = ttk.Label(top, text="Status: Disconnected", foreground="red")
        self.status_label.pack(side=tk.LEFT, padx=(6, 0))

        # Main area: 3 columns
        main = ttk.Frame(self)
        main.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=8, pady=8)

        # Left: grouped sliders
        left = ttk.Frame(main)
        left.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        # We'll create three LabelFrames in order to preserve column order:
        # Index (IDs 1,2), Pinkie (3,6), Thumb (5,4,7)
        groups = [
            ("Index", [1, 2]),
            ("Pinkie", [3, 6]),
            ("Thumb", [5, 4, 7])
        ]

        # Container to hold group frames horizontally
        groups_container = ttk.Frame(left)
        groups_container.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        for g_idx, (g_name, ids) in enumerate(groups):
            labelf = ttk.LabelFrame(groups_container, text=g_name, padding=(6,6))
            labelf.grid(row=0, column=g_idx, padx=8, pady=4, sticky="n")
            # inside each label frame, create a column per id (in order)
            for col_i, sid in enumerate(ids):
                col = ttk.Frame(labelf, relief=tk.RIDGE, borderwidth=1, padding=(6,6))
                col.grid(row=0, column=col_i, padx=6, pady=6, sticky="n")

                phal = PHALANGE_NAMES.get(sid, "")
                ttk.Label(col, text=f"ID {sid}\n{phal}", font=("TkDefaultFont", 10, "bold")).pack(anchor="n")

                min_lim, max_lim = RELATIVE_LIMITS.get(sid, (-90.0, 90.0))
                var = tk.DoubleVar(value=0.0)
                self.offset_vars[sid] = var

                scale = tk.Scale(col, from_=max_lim, to=min_lim,
                                 variable=var, resolution=0.5, length=300, orient=tk.VERTICAL,
                                 command=lambda v, _sid=sid: self._on_slider_move(_sid, float(v)))
                scale.pack(side=tk.TOP, fill=tk.Y, expand=True)
                self.scales[sid] = scale

                abs_label = ttk.Label(col, text=f"Target: {self._compute_abs(sid, 0.0):.1f}°")
                abs_label.pack(anchor="s", pady=(6,0))
                self.abs_labels[sid] = abs_label

                cur_label = ttk.Label(col, text="Current: ---")
                cur_label.pack(anchor="s")
                self.cur_labels[sid] = cur_label

                ttk.Label(col, text=f"Rel limits: [{min_lim}, {max_lim}]").pack(anchor="s", pady=(4,0))

        # Middle: vertical buttons column
        mid = ttk.Frame(main, width=180)
        mid.pack(side=tk.LEFT, fill=tk.Y, padx=(12,16))

        # Large button style using tk.Button so bg colors apply consistently
        btn_font = ("TkDefaultFont", 11, "bold")
        btn_opts = {"width": 14, "height": 2, "font": btn_font}

        # Execute (single-step)
        self.btn_execute = tk.Button(mid, text="Execute\n(Step)", bg="#d9534f", fg="white",
                                     command=self._start_execute, **btn_opts)
        self.btn_execute.pack(side=tk.TOP, pady=(6,10))

        # Home
        self.btn_home = tk.Button(mid, text="Home", bg="#5cb85c", fg="white",
                                  command=self._go_home, **btn_opts)
        self.btn_home.pack(side=tk.TOP, pady=10)

        # Save step
        self.btn_save_step = tk.Button(mid, text="Save step", bg="#0275d8", fg="white",
                                       command=self._save_step, **btn_opts)
        self.btn_save_step.pack(side=tk.TOP, pady=10)

        # Remove selected
        self.btn_remove = tk.Button(mid, text="Remove\nSelected", bg="#f0ad4e", fg="black",
                                    command=self._remove_selected, **btn_opts)
        self.btn_remove.pack(side=tk.TOP, pady=10)

        # Snapshot
        self.btn_snapshot = tk.Button(mid, text="Snapshot", bg="#6f42c1", fg="white",
                                      command=self._snapshot_positions, **btn_opts)
        self.btn_snapshot.pack(side=tk.TOP, pady=10)

        # Right: sequence (top) and log (bottom)
        right = ttk.Frame(main)
        right.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        # Sequence tree
        seq_panel = ttk.Frame(right)
        seq_panel.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        ttk.Label(seq_panel, text="Recorded sequence:", font=("TkDefaultFont", 10, "bold")).pack(anchor="nw", pady=(2,4))

        cols = ["step"] + [f"ID{sid}" for sid in SERVO_IDS]
        self.tree = ttk.Treeview(seq_panel, columns=cols, show="headings", height=12)
        self.tree.heading("step", text="step")
        self.tree.column("step", width=60, anchor="center")
        for sid in SERVO_IDS:
            colname = f"ID{sid}"
            self.tree.heading(colname, text=colname)
            self.tree.column(colname, width=70, anchor="center")
        self.tree.pack(side=tk.TOP, fill=tk.BOTH, expand=True, padx=6, pady=(0,6))

        # Log (below sequence)
        log_panel = ttk.Frame(right)
        log_panel.pack(side=tk.TOP, fill=tk.BOTH, expand=True)

        ttk.Label(log_panel, text="Log:", font=("TkDefaultFont", 10, "bold")).pack(anchor="nw", pady=(2,4))
        self.log_text = tk.Text(log_panel, height=12, wrap="word", state=tk.DISABLED)
        self.log_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        scrollbar = ttk.Scrollbar(log_panel, command=self.log_text.yview)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        self.log_text.configure(yscrollcommand=scrollbar.set)

    # ---------------- logging ----------------
    def _log(self, *parts):
        s = " ".join(str(p) for p in parts)
        ts = time.strftime("%H:%M:%S")
        line = f"[{ts}] {s}\n"
        try:
            self.log_text.configure(state=tk.NORMAL)
            self.log_text.insert(tk.END, line)
            self.log_text.see(tk.END)
            self.log_text.configure(state=tk.DISABLED)
        except Exception:
            print(line, end="")

    # ---------------- ports/status ----------------
    def _refresh_ports(self):
        ports = list_available_ports()
        self.port_combo['values'] = ports if ports else []
        if ports and self.port_var.get() not in ports:
            self.port_var.set(ports[0])
        self._log("Available ports:", ports or "<none>")

    def _update_status(self):
        if not self.connected:
            self.status_label.configure(text="Status: Disconnected", foreground="red")
        else:
            self.status_label.configure(text="Status: Connected (Real)", foreground="green")

    # ---------------- connect/disconnect ----------------
    def _connect_real(self):
        if not SDK_AVAILABLE:
            self._log("[ERROR] SDK not available. Install vassar_feetech_servo_sdk.")
            return
        port = self.port_var.get()
        servo_type = self.type_var.get()
        def bg_connect():
            try:
                ctrl = ServoController(servo_ids=SERVO_IDS, servo_type=servo_type, port=port)
                ctrl.connect()
                self.after(0, self._on_connected, ctrl)
            except Exception as e:
                self.after(0, self._on_connect_failed, e)
        threading.Thread(target=bg_connect, daemon=True).start()
        self._log("Attempting to connect (Real) to", port, "...")

    def _on_connected(self, controller):
        self.controller = controller
        self.connected = True
        self._update_status()
        self._log("Connected to real controller:", type(controller).__name__)
        self._read_positions_async()

    def _on_connect_failed(self, exc):
        self._log("[ERROR] Real connect failed:", exc)
        self.connected = False
        self.controller = None
        self._update_status()

    def _disconnect(self):
        if self.controller is None:
            self._log("Not connected.")
            return
        try:
            self.controller.disconnect()
        except Exception as e:
            self._log("Disconnect error:", e)
        finally:
            self.controller = None
            self.connected = False
            self._update_status()
            self._log("Disconnected.")

    # ---------------- basic helpers ----------------
    def _compute_abs(self, sid: int, offset: float) -> float:
        base = INITIAL_POSITIONS_DEG.get(sid, 0.0)
        return clamp_deg(base + offset)

    # ---------------- interactive slider handling ----------------
    def _on_slider_move(self, sid: int, new_val: float):
        min_lim, max_lim = RELATIVE_LIMITS[sid]
        val = float(new_val)
        if val < min_lim:
            val = min_lim
            self.offset_vars[sid].set(val)
        if val > max_lim:
            val = max_lim
            self.offset_vars[sid].set(val)
        abs_val = self._compute_abs(sid, val)
        self.abs_labels[sid].configure(text=f"Target: {abs_val:.1f}°")

        # debounce
        after_id = self._debounce_after_ids.get(sid)
        if after_id is not None:
            try:
                self.after_cancel(after_id)
            except Exception:
                pass
        aid = self.after(SLIDER_DEBOUNCE_MS, lambda _sid=sid: self._send_single_servo(_sid))
        self._debounce_after_ids[sid] = aid

    def _send_single_servo(self, sid: int):
        self._debounce_after_ids[sid] = None
        if not self.connected or self.controller is None:
            self._log(f"[WARN] Not connected: cannot send servo {sid}.")
            return
        offset = float(self.offset_vars[sid].get())
        target_deg = self._compute_abs(sid, offset)
        ticks_map = {sid: deg_to_ticks(target_deg)}
        speed = int(self.speed_var.get())
        accel = int(self.accel_var.get())
        def bg_write():
            try:
                res = self.controller.write_position(ticks_map, speed=speed, acceleration=accel)
                self._log(f"write_position (ID {sid}) -> ticks {ticks_map[sid]} ; res: {res}")
                self.after(0, lambda: self.cur_labels[sid].configure(text=f"Current: {target_deg:.1f}°"))
            except Exception as e:
                self._log("[ERROR] write_position exception (ID {}):".format(sid), e)
        threading.Thread(target=bg_write, daemon=True).start()

    # ---------------- sequence management ----------------
    def _save_step(self):
        step = {sid: self._compute_abs(sid, float(self.offset_vars[sid].get())) for sid in SERVO_IDS}
        self.sequence.append(step)
        self._tree_insert_step(len(self.sequence), step)
        self._log("Saved step", len(self.sequence), step)

    def _tree_insert_step(self, idx: int, step: Dict[int, float]):
        vals = [str(idx)] + [f"{step[sid]:.1f}" for sid in SERVO_IDS]
        self.tree.insert("", "end", iid=str(idx-1), values=vals)

    def _refresh_tree(self):
        for item in self.tree.get_children():
            self.tree.delete(item)
        for i, step in enumerate(self.sequence, start=1):
            self._tree_insert_step(i, step)

    def _remove_selected(self):
        sel = self.tree.selection()
        if not sel:
            self._log("[INFO] No selection to remove.")
            return
        indices = sorted([int(iid) for iid in sel], reverse=True)
        for iid in indices:
            idx0 = int(iid)
            if 0 <= idx0 < len(self.sequence):
                removed = self.sequence.pop(idx0)
                self._log(f"Removed step {idx0+1}: {removed}")
        self._refresh_tree()
        self._exec_next_index = min(self._exec_next_index, len(self.sequence))

    # ---------------- Home ----------------
    def _go_home(self):
        for sid in SERVO_IDS:
            self.offset_vars[sid].set(0.0)
            self.abs_labels[sid].configure(text=f"Target: {self._compute_abs(sid, 0.0):.1f}°")
        targets = {sid: self._compute_abs(sid, 0.0) for sid in SERVO_IDS}
        if not self.connected or self.controller is None:
            self._log("[WARN] Not connected; Home applied locally only.")
            return
        ticks_map = build_ticks_map(targets)
        speed = int(self.speed_var.get())
        accel = int(self.accel_var.get())
        def bg_home():
            try:
                res = self.controller.write_position(ticks_map, speed=speed, acceleration=accel)
                self._log("Home write_position response:", res)
                for sid, deg in targets.items():
                    self.after(0, lambda _sid=sid, _deg=deg: self.cur_labels[_sid].configure(text=f"Current: {_deg:.1f}°"))
            except Exception as e:
                self._log("[ERROR] Home write_position exception:", e)
        threading.Thread(target=bg_home, daemon=True).start()

    # ---------------- execution ----------------
    def _start_execute(self):
        with self._exec_lock:
            if self._is_executing:
                self._log("[WARN] Already executing.")
                return
            if not self.sequence:
                self._log("[WARN] Sequence empty.")
                return
            if not self.connected or self.controller is None:
                self._log("[ERROR] Not connected; cannot execute.")
                return
            if self._exec_next_index >= len(self.sequence):
                self._log("[INFO] No more steps. Resetting pointer to 0.")
                self._exec_next_index = 0
                return
            self._exec_stop_event.clear()
            self._is_executing = True
            # disable buttons while executing to avoid conflicts
            self._set_buttons_state("disabled")
            idx_to_exec = self._exec_next_index
            self._exec_thread = threading.Thread(target=self._exec_step_thread, args=(idx_to_exec,), daemon=True)
            self._exec_thread.start()
            self._log(f"Executing step {idx_to_exec+1}...")

    def _exec_step_thread(self, idx0: int):
        try:
            if idx0 < 0 or idx0 >= len(self.sequence):
                self._log("[EXEC][ERROR] Index out of range:", idx0)
                return
            step = self.sequence[idx0]
            self._log(f"[EXEC] Sending step {idx0+1}: { {k: f'{v:.1f}' for k,v in step.items()} }")
            ticks_map = build_ticks_map(step)
            speed = int(self.speed_var.get())
            accel = int(self.accel_var.get())
            try:
                res = self.controller.write_position(ticks_map, speed=speed, acceleration=accel)
                self._log(f"[EXEC] write_position result for step {idx0+1}:", res)
            except Exception as e:
                self._log("[EXEC][ERROR] write_position exception:", e)
                return

            reached = wait_until_reached(self.controller, step,
                                         tol_deg=STEP_TOLERANCE_DEG,
                                         timeout_s=STEP_TIMEOUT_S,
                                         poll_interval=0.15,
                                         stop_event=self._exec_stop_event)
            if not reached:
                if self._exec_stop_event.is_set():
                    self._log("[EXEC] Stopped while waiting for step.")
                else:
                    # Timeout occurred — reset the execution pointer so planner restarts from step 0 next time
                    self._log(f"[EXEC] Step {idx0+1} NOT reached within timeout ({STEP_TIMEOUT_S}s). Resetting planner pointer to 0.")
                    # reset pointer
                    self._exec_next_index = 0
            else:
                self._log(f"[EXEC] Step {idx0+1} reached.")
                self._exec_next_index = min(idx0 + 1, len(self.sequence))
        finally:
            with self._exec_lock:
                self._is_executing = False
                self._exec_thread = None
                self._exec_stop_event.clear()
                self._set_buttons_state("normal")
                self._log("Execution finished/aborted.")

    def _set_buttons_state(self, state: str):
        # state: "disabled" or "normal"
        widgets = [self.btn_execute, self.btn_home, self.btn_save_step, self.btn_remove, self.btn_snapshot]
        for w in widgets:
            try:
                if state == "disabled":
                    w.configure(state=tk.DISABLED)
                else:
                    w.configure(state=tk.NORMAL)
            except Exception:
                pass

    # ---------------- snapshot ----------------
    def _snapshot_positions(self):
        if not self.connected or self.controller is None:
            self._log("[WARN] Not connected; cannot snapshot.")
            return
        def bg_snap():
            try:
                vals = read_positions_batch(self.controller, SERVO_IDS)
                def apply():
                    for sid, deg in vals.items():
                        if deg is None:
                            continue
                        base = INITIAL_POSITIONS_DEG.get(sid, 0.0)
                        desired_offset = deg - base
                        min_lim, max_lim = RELATIVE_LIMITS.get(sid, (-90.0, 90.0))
                        if desired_offset < min_lim:
                            desired_offset = min_lim
                        if desired_offset > max_lim:
                            desired_offset = max_lim
                        self.offset_vars[sid].set(desired_offset)
                        self.abs_labels[sid].configure(text=f"Target: {self._compute_abs(sid, desired_offset):.1f}°")
                        self.cur_labels[sid].configure(text=f"Current: {deg:.1f}°")
                    self._log("Snapshot applied:", {k: (None if v is None else f"{v:.1f}°") for k,v in vals.items()})
                self.after(0, apply)
            except Exception as e:
                self._log("[ERROR] Snapshot failed:", e)
        threading.Thread(target=bg_snap, daemon=True).start()

    # ---------------- read helpers ----------------
    def _read_positions_async(self):
        def bg_read():
            try:
                vals = read_positions_batch(self.controller, SERVO_IDS)
                self.after(0, lambda: self._apply_read_values(vals))
                self._log("Initial read positions:", {k: (None if v is None else f"{v:.1f}°") for k, v in vals.items()})
            except Exception as e:
                self._log("[ERROR] Initial read failed:", e)
        threading.Thread(target=bg_read, daemon=True).start()

    def _apply_read_values(self, values: Dict[int, Optional[float]]):
        for sid, deg in values.items():
            self.cur_labels[sid].configure(text=(f"Current: {deg:.1f}°" if deg is not None else "Current: ---"))

    # ---------------- periodic poll ----------------
    def _periodic_poll(self):
        if self.auto_poll.get() and self.connected and self.controller is not None:
            def bg_poll():
                try:
                    vals = read_positions_batch(self.controller, SERVO_IDS)
                    self.after(0, lambda: self._apply_read_values(vals))
                except Exception as e:
                    self._log("[WARN] Poll read error:", e)
            threading.Thread(target=bg_poll, daemon=True).start()
        self.after(GUI_POLL_MS, self._periodic_poll)

# ---------------- main ----------------
if __name__ == "__main__":
    app = ServoGUI()
    app.mainloop()


Checking servo phase values...
Motor 1: Phase already 0 ✓
Motor 2: Phase already 0 ✓
Motor 3: Phase already 0 ✓
Motor 4: Phase already 0 ✓
Motor 5: Phase already 0 ✓
Motor 6: Phase already 0 ✓
Motor 7: Phase already 0 ✓
Failed to read mode for motor 6
Sync write failed: [TxRxResult] Protocol does not support this function!
Failed to read mode for motor 6
Sync write failed: [TxRxResult] Protocol does not support this function!
Failed to read mode for motor 1
Failed to read mode for motor 2
Failed to read mode for motor 3
Failed to read mode for motor 4
Failed to read mode for motor 5
Failed to read mode for motor 6
Failed to read mode for motor 7
Sync write failed: [TxRxResult] Protocol does not support this function!
